# 시계열 데이터 다루기

In [ ]:
import os
import pandas as pd
import numpy as np
from plt_rcs import *
import hds

In [ ]:
dates = pd.date_range(start='2024-01-01', end='2025-12-31', name='date')
# DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
#                '2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
#                '2024-01-09', '2024-01-10',
#                ...
#                '2025-12-22', '2025-12-23', '2025-12-24', '2025-12-25',
#                '2025-12-26', '2025-12-27', '2025-12-28', '2025-12-29',
#                '2025-12-30', '2025-12-31'],
#               dtype='datetime64[ns]', name='date', length=731, freq='D')

## 시계열 요소 설정

In [ ]:
n = len(dates)

In [ ]:
# 완만한 증가 추세 설정
trend = np.linspace(100, 140, n)

In [ ]:
# 연간 계절성 설정
yearly = -20 * np.cos(2 * np.pi * np.arange(n) / 365)

In [ ]:
# 주간 계절성 설정
weekly = np.where(dates.dayofweek < 5, 10, -20)

In [ ]:
# 시드 고정
np.random.seed(1)

In [ ]:
# 불확실성 설정
noise = np.random.normal(loc=0, scale=3, size=n)

In [ ]:
# 문자열로 변환
dates = dates.astype(str)

## 가상의 시계열 데이터 생성

In [ ]:
df = pd.DataFrame(data={'date': dates, 'value': trend + yearly + weekly + noise})

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df['value'] = df['value'].round(2)

In [ ]:
sns.lineplot(data=df, x='date', y='value', color='0.8')
sns.lineplot(x=range(731), y=trend, color='red', linewidth=2)
plt.show()

## 누락 및 중복 시점 생성

In [ ]:
# 삭제할 날짜 리스트 생성
drop_dates = ['2024-01-05', '2024-02-10', '2024-03-20']

In [ ]:
# 선택한 행 삭저
df = df.loc[~df['date'].isin(drop_dates), :].copy()

In [ ]:
# 중복 추가 날짜 시르트 생성
dup_dates = ['2024-01-15', '2024-03-31', '2024-04-25']

In [ ]:
# 선택한 행 삭저
dup_rows = df.loc[df['date'].isin(dup_dates), :].copy()

## 가상의 시계열 데이터 변형

In [ ]:
# 행 방향으로 결합 후 재할당
df = pd.concat(objs=[df, dup_rows], ignore_index=True)

In [ ]:
# 순서를 임의로 섞어 정렬되지 않은 상태로 만들기
df = df.sample(frac=1, random_state=1).reset_index(drop=True)

In [ ]:
# 일부 값을 결측으로 처리
df.loc[df.sample(n=5, random_state=1).index, 'value'] = np.nan

In [ ]:
df.info()

## 날짜시간 데이터로 변환

In [ ]:
# 날짜시간형으로 변환
df['date'] = pd.to_datetime(arg=df['date'], errors='coerce')

In [ ]:
df.dtypes

In [ ]:
# 인덱스 설정 후 오름차순 정렬하여 재할당
df = df.set_index(keys='date').sort_index()

In [ ]:
df.head()

## 날짜 파생변수 생성

In [ ]:
df['year'] = df.index.year

In [ ]:
df['quarter'] = df.index.quarter

In [ ]:
df['month'] = df.index.month

In [ ]:
df['day'] = df.index.day

In [ ]:
df['dayofweek'] = df.index.day_name(locale='ko_KR')

In [ ]:
df['is_weekend'] = (df.index.dayofweek >= 5)

In [ ]:
df.head(10)

In [ ]:
df = df.filter(items=['value'])

In [ ]:
df.head()

## 시계열 데이터의 인덱싱 및 슬라이싱

In [ ]:
df.loc['2024-01-01', :]

In [ ]:
df.loc['2024-01-01':'2024-01-10', :]

In [ ]:
df.loc[['2024-01-01', '2025-01-01'], :]

In [ ]:
df.loc[df.index.month == 1, :]

## 중복 시점 확인

In [ ]:
# 중복 여부 마스크 생성
dup_mask = df.index.duplicated(keep=False)

In [ ]:
dup_rows = df.loc[dup_mask]

In [ ]:
dup_rows
#               value
# date	
# 2024-01-15	94.75
# 2024-01-15	94.75
# 2024-03-31	88.10
# 2024-03-31	88.10
# 2024-04-25	120.32
# 2024-04-25	120.32

- 중간에 있는 행 선택
- nth 메서드 괄호안에 정수 인덱스를 지정하면 n+1 번째 행을 반환
    - 0 : first
    - -1 : last
- groupby 메서드의 sort 매개변수에 전달되는 인수는 True여야함

In [ ]:
dup_rows.groupby(dup_rows.index).nth(0)

In [ ]:
# 그룹 설정 후 편균 계산하여 재할당
df = df.groupby(df.index)['value'].mean()

## 누락 시점 확인

In [ ]:
df.shape

In [ ]:
# 기대 인덱스 생성 후 할당
full_idx = pd.date_range(df.index.min(), df.index.max())

In [ ]:
# 누락된 인덱스를 생성 후 할당
miss_idx = full_idx.difference(df.index)

In [ ]:
miss_idx

In [ ]:
# 누락된 인덱스를 추가 후 재할당
df = df.reindex(index=full_idx)

## 결측값 처리

In [ ]:
# 결측행 확인
df.loc[df.isna()]
# 2024-01-05   NaN
# 2024-02-10   NaN
# 2024-03-20   NaN
# 2024-06-15   NaN
# 2024-09-09   NaN
# 2024-12-28   NaN
# 2025-10-10   NaN
# 2025-12-28   NaN
# Name: value, dtype: float64

In [ ]:
df = df.to_frame()
df.index.name = 'date'

In [ ]:
df['value_ffill'] = df['value'].ffill()

In [ ]:
df['value_linear'] = df['value'].interpolate(method='linear')

In [ ]:
df['value_time'] = df['value'].interpolate(method='time')

In [ ]:
df.loc['2024-01-04':'2024-01-10', :]

In [ ]:
sr = pd.Series(data=[1, 2, np.nan, 6, 7])
sr.index = pd.date_range('2026-01-01', '2026-01-05')

In [ ]:
sr.interpolate(method='linear')

In [ ]:
sr.index = pd.to_datetime(['2026-01-01', '2026-01-02', '2026-01-03', '2026-01-06', '2026-01-07'])

In [ ]:
sr.interpolate(method='time')

In [ ]:
df.columns

In [ ]:
# 불필요한 컬럼 모두 삭제
df = df.drop(columns=['value', 'value_ffill', 'value_time'])

In [ ]:
# 남은 열이름 변경
df = df.rename(columns={'value_linear': 'value'})

In [ ]:
df.head()

## 시차 변수 생성

In [ ]:
# 아래로 값 이동
df['lag_1'] = df['value'].shift(periods=1)
df['lag_2'] = df['value'].shift(periods=2)
df['lag_3'] = df['value'].shift(periods=3)

In [ ]:
df.head()

In [ ]:
df.corr()

## 이동 평균 계산

In [ ]:
df['roll_7_avg'] = df['value'].rolling(window=7, min_periods=1).mean()
df['roll_7_std'] = df['value'].rolling(window=7, min_periods=1).std()

In [ ]:
df.head()

In [ ]:
df['expand_avg'] = df['value'].expanding().mean()

In [ ]:
df.head(10)

## 변화량 기반 변수 생성

In [ ]:
# 전일 대비 차분 계산
df['diff_1'] = df['value'].diff(periods=1)

In [ ]:
df.head()

In [ ]:
# 전일 대비 증감률 계산
df['pent_1'] = df['value'].pct_change(periods=1)

In [ ]:
df.head()

In [ ]:
df = df.filter(like='value')

In [ ]:
df.head()

## 시간 단위 재구성

In [ ]:
df.loc['2024-01-01':'2024-01-07', 'value'].sum()

In [ ]:
# 주 단위로 합계 계산
df.resample(rule='W-SUN')['value'].sum()

In [ ]:
# 주 단위 합계 4주 이동 평균 계산
df.resample(rule='W-SUN')['value'].sum().rolling(window=4).mean()

In [ ]:
# 특정 요일 선택하여 반환
df.asfreq(freq='W-FRI')

## 가상의 시계열 데이터 시각화

In [ ]:
sns.lineplot(x=df.index, y=df['value'], color='royalblue', linewidth=1)
plt.show()

## 시계열 분해

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
result = seasonal_decompose(x=df['value'], model='additive', period=7)

In [ ]:
result.plot();

## 외부 파일로 저장

In [ ]:
os.getcwd()

In [ ]:
os.chdir('../../data')

In [ ]:
df.to_pickle(path='Synthetic_Time_Series.pkl')